In [1]:
#Our data currently looks like:

#Source:
#question: Does vitamin D improve...
#context: Vitamin D has been shown...

#Target:
#The study suggests that vitamin D...

#We need to convert:

#Text
# ↓
#Tokens
# ↓
#Token IDs
# ↓
#Tensors

#For example:

#"vitamin D improves health"
#        ↓
#["vitamin", "D", "improves", "health"]
#        ↓
#[125, 42, 891, 317]

In [2]:
from pathlib import Path
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

# Processed data directory
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load datasets
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (800, 12)
Validation: (100, 12)
Test: (100, 12)


In [3]:
#we loading all three?
#Because eventually:
#Train → build vocabulary + train model
#Validation → tune/check model during training
#Test → final evaluation
#And an important rule:
#Our vocabulary will be built using training data only.

In [4]:
# Trying our own tokenizer:
sample_text = train_df["source"].iloc[0]

tokens = sample_text.split()

print("Original text:")
print(sample_text[:500])

print("\nFirst 30 tokens:")
print(tokens[:30])

print("\nNumber of tokens:", len(tokens))

Original text:
question: Visceral adipose tissue area measurement at a single level: can it represent visceral adipose tissue volume? context: Measurement of visceral adipose tissue (VAT) needs to be accurate and sensitive to change for risk monitoring. The purpose of this study is to determine the CT slice location where VAT area can best reflect changes in VAT volume and body weight. 60 plain abdominal CT images from 30 males [mean age (range) 51 (41-68) years, mean body weight (range) 71.1 (101.9-50.9) kg] 

First 30 tokens:
['question:', 'Visceral', 'adipose', 'tissue', 'area', 'measurement', 'at', 'a', 'single', 'level:', 'can', 'it', 'represent', 'visceral', 'adipose', 'tissue', 'volume?', 'context:', 'Measurement', 'of', 'visceral', 'adipose', 'tissue', '(VAT)', 'needs', 'to', 'be', 'accurate', 'and', 'sensitive']

Number of tokens: 155


In [5]:
# Understandings from above i can see the punctution are also there so its not enough:
# regular expression can separate words and punctuation

In [6]:
import re

sample = "Vitamin D improves bone health. Does it work?"

tokens = re.findall(r"\w+|[^\w\s]", sample)

print(tokens)

['Vitamin', 'D', 'improves', 'bone', 'health', '.', 'Does', 'it', 'work', '?']


In [7]:
# Now its working well lets try it with .split() and regex tokenization

In [8]:
sample_text = train_df["source"].iloc[0]

# Simple whitespace tokenization
split_tokens = sample_text.split()

# Regex tokenization
regex_tokens = re.findall(r"\w+|[^\w\s]", sample_text)

print("Whitespace tokens:", len(split_tokens))
print("Regex tokens:", len(regex_tokens))

print("\nFirst 40 regex tokens:")
print(regex_tokens[:40])

Whitespace tokens: 155
Regex tokens: 196

First 40 regex tokens:
['question', ':', 'Visceral', 'adipose', 'tissue', 'area', 'measurement', 'at', 'a', 'single', 'level', ':', 'can', 'it', 'represent', 'visceral', 'adipose', 'tissue', 'volume', '?', 'context', ':', 'Measurement', 'of', 'visceral', 'adipose', 'tissue', '(', 'VAT', ')', 'needs', 'to', 'be', 'accurate', 'and', 'sensitive', 'to', 'change', 'for', 'risk']


In [9]:
#With .split():
#"question:" one token

#With regex: "question:"
#two tokens:"question", ":"

# Much controll but some hypens and other symbol are there

#Checking where simple regex tokenizer will not work

In [10]:
biomedical_examples = [
    "ΔΨm",
    "A. madagascariensis",
    "nDNA",
    "IL-6",
    "COVID-19",
    "p-value",
    "10.5",
    "α-synuclein"
]

for text in biomedical_examples:
    tokens = re.findall(r"\w+|[^\w\s]", text)
    print(f"{text:20} → {tokens}")

ΔΨm                  → ['ΔΨm']
A. madagascariensis  → ['A', '.', 'madagascariensis']
nDNA                 → ['nDNA']
IL-6                 → ['IL', '-', '6']
COVID-19             → ['COVID', '-', '19']
p-value              → ['p', '-', 'value']
10.5                 → ['10', '.', '5']
α-synuclein          → ['α', '-', 'synuclein']


In [11]:
# In above suitaion i can clearly see symbols hypen are splited but we cannot bilindly tokeniez every thing
# building a tokenizer function

In [12]:
def tokenize(text):
    """
    Convert text into a list of tokens.
    """
    text = text.strip()
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens

In [13]:
sample = "IL-6 and COVID-19 were studied in 10.5 patients."

tokens = tokenize(sample)

print(tokens)

['IL', '-', '6', 'and', 'COVID', '-', '19', 'were', 'studied', 'in', '10', '.', '5', 'patients', '.']


In [14]:
# Now i can clearly sees that IL - 6 are splitted in better scientify way the way it should be
# vocabulary needs to know words from both sides from source vocab and from target vocab

In [15]:
# Counting tokens in the training data

from collections import Counter

MIN_FREQUENCY = 2

token_counter = Counter()

# Count tokens from SOURCE
for text in train_df["source"]:
    tokens = tokenize(text)
    token_counter.update(tokens)

# Count tokens from TARGET
for text in train_df["target"]:
    tokens = tokenize(text)
    token_counter.update(tokens)

print("Total unique tokens:", len(token_counter))

print("\nMost common 20 tokens:")
print(token_counter.most_common(20))

Total unique tokens: 15128

Most common 20 tokens:
[('.', 14273), (',', 8370), ('of', 7845), ('the', 7583), ('and', 6649), ('-', 5267), (')', 5234), ('(', 5212), ('in', 5007), ('to', 3732), ('with', 3042), ('a', 2895), ('was', 2516), ('were', 2513), ('%', 2482), (':', 2349), ('for', 2251), ('patients', 2232), ('0', 1952), ('The', 1654)]


In [16]:
#total unique tokens and most common tokens

In [17]:
#Building the vocabulary:- the collection of unique tokens our model knows
#we will build are vocab from our training set not from training + validation + test b/c that will allow our information
#from our evaluation data to influence the tokenizer after that Validation and test text will use the same vocabulary

In [18]:
#TRAIN TEXT
#    ↓
#Tokenize
#    ↓
#Unique tokens
#    ↓
#Vocabulary
#    ↓
#Token IDs

In [19]:
# We necesssary not use those token whose frequency is less than lets say 5 or 6 to not make our vocab large
# before doing anything lets see what are the frequencys are

In [20]:
frequencies = list(token_counter.values())

print("Total unique tokens:", len(frequencies))
print("Tokens appearing once:", sum(freq == 1 for freq in frequencies))
print("Tokens appearing at least twice:", sum(freq >= 2 for freq in frequencies))
print("Tokens appearing at least 5 times:", sum(freq >= 5 for freq in frequencies))
print("Tokens appearing at least 10 times:", sum(freq >= 10 for freq in frequencies))

Total unique tokens: 15128
Tokens appearing once: 5596
Tokens appearing at least twice: 9532
Tokens appearing at least 5 times: 5021
Tokens appearing at least 10 times: 2796


In [21]:
# We will first see with min_freq with 2 b/c many tokens are usefull in medical terms so it will hamper our data

In [22]:
# Building vocabulary

MIN_FREQUENCY = 2

vocab_tokens = [
    token
    for token, count in token_counter.items()
    if count >= MIN_FREQUENCY
]

print("Vocabulary size before special tokens:", len(vocab_tokens))
print("First 30 vocabulary tokens:")
print(vocab_tokens[:30])


Vocabulary size before special tokens: 9532
First 30 vocabulary tokens:
['question', ':', 'adipose', 'tissue', 'area', 'measurement', 'at', 'a', 'single', 'level', 'can', 'it', 'represent', 'visceral', 'volume', '?', 'context', 'Measurement', 'of', '(', 'VAT', ')', 'needs', 'to', 'be', 'accurate', 'and', 'sensitive', 'change', 'for']


In [23]:
# Before Assigning the id's we will make a specials tokens
# for Token not found in vocab
# end of sequence
# start of sequence
# Seprate question from context 
# padding shorter sequence

In [24]:
# Adding special tokens

SPECIAL_TOKENS = ["<PAD>", "<UNK>", "<BOS>", "<EOS>", "<SEP>"]

token_to_id = {}

# Assign IDs to special tokens first
for idx, token in enumerate(SPECIAL_TOKENS):
    token_to_id[token] = idx

# Add normal vocabulary tokens
for token in vocab_tokens:
    if token not in token_to_id:
        token_to_id[token] = len(token_to_id)

print("Vocabulary size:", len(token_to_id))

print("\nSpecial token IDs:")
for token in SPECIAL_TOKENS:
    print(f"{token:6} → {token_to_id[token]}")

Vocabulary size: 9537

Special token IDs:
<PAD>  → 0
<UNK>  → 1
<BOS>  → 2
<EOS>  → 3
<SEP>  → 4


In [25]:
#Token ID to Token:-

#We have: token to id
#which means:"patient" → 123

#But later,  the Transformer generates:[2, 45, 891, 317, 3]

#we need to convert those IDs back into text
#So we will also change: id to token

# Basically we do encoding and decoding.
# or we can say reverse mapping.

In [26]:
#Creating Token to id:
SPECIAL_TOKENS = ["<PAD>", "<UNK>", "<BOS>", "<EOS>", "<SEP>"]

token_to_id = {}

for idx, token in enumerate(SPECIAL_TOKENS):
    token_to_id[token] = idx

for token in vocab_tokens:
    if token not in token_to_id:
        token_to_id[token] = len(token_to_id)

print("Vocabulary size:", len(token_to_id))

print("\nSpecial token IDs:")
for token in SPECIAL_TOKENS:
    print(f"{token:6} → {token_to_id[token]}")

Vocabulary size: 9537

Special token IDs:
<PAD>  → 0
<UNK>  → 1
<BOS>  → 2
<EOS>  → 3
<SEP>  → 4


In [27]:
id_to_token = {
    idx: token
    for token, idx in token_to_id.items()
}

print("ID → Token examples:")

for idx in range(10):
    print(f"{idx} → {id_to_token[idx]}")

ID → Token examples:
0 → <PAD>
1 → <UNK>
2 → <BOS>
3 → <EOS>
4 → <SEP>
5 → question
6 → :
7 → adipose
8 → tissue
9 → area


In [28]:
# Now Coverting tokens to ids:  

In [29]:
# Converting tokens into token IDs

sample_tokens = ["question", ":", "Visceral", "adipose", "tissue"]

sample_ids = [
    token_to_id.get(token, token_to_id["<UNK>"])
    for token in sample_tokens
]

print("Tokens:")
print(sample_tokens)

print("\nToken IDs:")
print(sample_ids)

Tokens:
['question', ':', 'Visceral', 'adipose', 'tissue']

Token IDs:
[5, 6, 1, 7, 8]


In [30]:
#we can see here question has id 5 means: known vocab word and visceral got token id 1 means : not in vocab
#why this happend with visceral b/c it occur only once and we set the min freq to 2
#limitations of our simple word-level tokenizer it dosent have unique token id for every possible biomedical word

In [31]:
# Now lets creating a function for text to tokens to token ids we are creating a resuseble function so that we dosent have to write the whole code again and agin

In [32]:
# Function to encode text into token IDs

def encode_text(text):
    tokens = tokenize(text)
    
    token_ids = [
        token_to_id.get(token, token_to_id["<UNK>"])
        for token in tokens
    ]
    
    return token_ids

In [33]:
#Checking three things:
#Normal vocab tokens to their IDs then, Unknown tokens to <UNK> ID then Punctuation to its corresponding ID

In [34]:
# Testing the text encoder

sample_text = "question: What is cancer?"

encoded = encode_text(sample_text)

print("Original text:")
print(sample_text)

print("\nEncoded token IDs:")
print(encoded)

Original text:
question: What is cancer?

Encoded token IDs:
[5, 6, 1, 42, 797, 20]


In [35]:
# we can again see that what got 1 unk not in vocab

In [36]:
# now we will do padding b/c of the sequence length varies from sentence to sentence 
# why we need padding b/c when we create a batch for the Transformer, we want a rectangular tensor
# padding allows us to add empty position so that all sequence have same length
# To control the transformer from not treating our pad or 0 a real word we will do a attention masking which help us to ignore that value

In [37]:
# Checking padding on low sequence first :-

sequences = [
    [5, 6, 42, 797],
    [5, 6, 31, 88, 19, 72],
    [5, 6, 17]
]

max_length = max(len(seq) for seq in sequences)

padded_sequences = [
    seq + [token_to_id["<PAD>"]] * (max_length - len(seq))
    for seq in sequences
]

print("Original sequences:")
print(sequences)

print("\nPadded sequences:")
print(padded_sequences)

Original sequences:
[[5, 6, 42, 797], [5, 6, 31, 88, 19, 72], [5, 6, 17]]

Padded sequences:
[[5, 6, 42, 797, 0, 0], [5, 6, 31, 88, 19, 72], [5, 6, 17, 0, 0, 0]]


In [38]:
# padding workes identifiy the longest sequence and padded other with same length

In [39]:
# Function to pad sequences

def pad_sequences(sequences, max_length=None):
    if max_length is None:
        max_length = max(len(seq) for seq in sequences)

    padded = [
        seq + [token_to_id["<PAD>"]] * (max_length - len(seq))
        for seq in sequences
    ]

    return padded

In [40]:
# Testing the padding function

padded_test = pad_sequences(sequences)

print("Padded sequences:")
print(padded_test)

Padded sequences:
[[5, 6, 42, 797, 0, 0], [5, 6, 31, 88, 19, 72], [5, 6, 17, 0, 0, 0]]


In [41]:
# same output means working fine

In [42]:
# Now chossing the max sequence length
# we cannot choose the max_length to unlimited b/c it will be wsted computation b/c of increase in tokensids 
# Source: 512 tokens
# Target: 128 tokens
# Why i choose it b/c of EDA max length  
# so 128 is a reasonable starting target limit.

# We have two things :
# Source = (question + context) which needs more room.
# Target = answer which can generally shorter.

In [43]:
# Maximum sequence lengths

SOURCE_MAX_LENGTH = 512
TARGET_MAX_LENGTH = 128

print("Source maximum length:", SOURCE_MAX_LENGTH)
print("Target maximum length:", TARGET_MAX_LENGTH)

Source maximum length: 512
Target maximum length: 128


In [44]:
#SOURCE TEXT
#    ↓
#tokenize()
#    ↓
#token IDs
#    ↓
#truncate to 512
#    ↓
#pad to 512

#for target 
#TARGET TEXT
#    ↓
#tokenize()
#    ↓
#token IDs
#    ↓
#truncate to 128
#    ↓
#pad to 128

In [45]:
# Testing source and target encoding on one training example

sample_source = train_df.iloc[0]["source"]
sample_target = train_df.iloc[0]["target"]

source_ids = encode_text(sample_source)
target_ids = encode_text(sample_target)

print("Source text:")
print(sample_source[:500])

print("\nSource token count:", len(source_ids))

print("\nTarget text:")
print(sample_target)

print("\nTarget token count:", len(target_ids))

Source text:
question: Visceral adipose tissue area measurement at a single level: can it represent visceral adipose tissue volume? context: Measurement of visceral adipose tissue (VAT) needs to be accurate and sensitive to change for risk monitoring. The purpose of this study is to determine the CT slice location where VAT area can best reflect changes in VAT volume and body weight. 60 plain abdominal CT images from 30 males [mean age (range) 51 (41-68) years, mean body weight (range) 71.1 (101.9-50.9) kg] 

Source token count: 196

Target text:
VAT area measurement at a single level 3 cm above the lower margin of the L3 vertebra is feasible and can reflect changes in VAT volume and body weight. Advances in knowledge: As VAT area at a CT slice 3cm above the lower margin of L3 can best reflect interval changes in VAT volume and body weight, VAT area measurement should be selected at this location.

Target token count: 70


In [46]:
# we can see thier is no truncation happen b/c the token count satisfied the limit but sequence where token count exceed the length the remaing one will truncate

In [47]:
# Applying truncation and padding to the sample

source_ids = source_ids[:SOURCE_MAX_LENGTH]
target_ids = target_ids[:TARGET_MAX_LENGTH]

source_padded = pad_sequences([source_ids], SOURCE_MAX_LENGTH)[0]
target_padded = pad_sequences([target_ids], TARGET_MAX_LENGTH)[0]

print("Source sequence length:", len(source_padded))
print("Target sequence length:", len(target_padded))

print("\nNumber of source PAD tokens:",
      source_padded.count(token_to_id["<PAD>"]))

print("Number of target PAD tokens:",
      target_padded.count(token_to_id["<PAD>"]))

Source sequence length: 512
Target sequence length: 128

Number of source PAD tokens: 316
Number of target PAD tokens: 58


In [48]:
# Original: 196 tokens , Maximum: 512 , Padding: 316 , Final: 512
# now preprocessing can handel short sequences by padding and long by truncating them

In [49]:
# Encoding the training source sequences

train_source_ids = []

for text in train_df["source"]:
    ids = encode_text(text)
    ids = ids[:SOURCE_MAX_LENGTH]
    ids = pad_sequences([ids], SOURCE_MAX_LENGTH)[0]
    train_source_ids.append(ids)

print("Number of training source sequences:", len(train_source_ids))
print("Length of first sequence:", len(train_source_ids[0]))

Number of training source sequences: 800
Length of first sequence: 512


## 

## Above we have done for Source 
## Now Process with Target (max_length=128)

In [50]:
# same process which we done for source : Tokenize to IDs to truncate to pad

In [51]:
# Encoding the training target sequences

train_target_ids = []

for text in train_df["target"]:
    ids = encode_text(text)
    ids = ids[:TARGET_MAX_LENGTH]
    ids = pad_sequences([ids], TARGET_MAX_LENGTH)[0]
    train_target_ids.append(ids)

print("Number of training target sequences:", len(train_target_ids))
print("Length of first target sequence:", len(train_target_ids[0]))

Number of training target sequences: 800
Length of first target sequence: 128


In [52]:
#SOURCE Question + Context:512 positions and for TARGET answer : 128 position  
#we will also check how much information is truncating             

In [53]:
# Checking how many sequences were truncated

source_truncated = 0
target_truncated = 0

for text in train_df["source"]:
    if len(encode_text(text)) > SOURCE_MAX_LENGTH:
        source_truncated += 1

for text in train_df["target"]:
    if len(encode_text(text)) > TARGET_MAX_LENGTH:
        target_truncated += 1

print("Source sequences truncated:", source_truncated)
print("Target sequences truncated:", target_truncated)

print("Source truncation percentage:",
      round(source_truncated / len(train_df) * 100, 2), "%")

print("Target truncation percentage:",
      round(target_truncated / len(train_df) * 100, 2), "%")

Source sequences truncated: 7
Target sequences truncated: 0
Source truncation percentage: 0.88 %
Target truncation percentage: 0.0 %


In [54]:
# Analyzing source token lengths

source_lengths = []

for text in train_df["source"]:
    source_lengths.append(len(encode_text(text)))

print("Minimum source length:", min(source_lengths))
print("Maximum source length:", max(source_lengths))

print("\nAverage source length:", round(sum(source_lengths) / len(source_lengths), 2))

print("\nSource lengths at important percentiles:")

for percentile in [50, 75, 90, 95, 99]:
    value = sorted(source_lengths)[int(len(source_lengths) * percentile / 100)]
    print(f"{percentile}th percentile: {value}")

Minimum source length: 78
Maximum source length: 598

Average source length: 285.83

Source lengths at important percentiles:
50th percentile: 281
75th percentile: 332
90th percentile: 386
95th percentile: 423
99th percentile: 510


### Now we are Using Same Vocab which we built from training data for Validation:-

In [55]:
# Encoding validation source sequences

val_source_ids = []

for text in val_df["source"]:
    ids = encode_text(text)
    ids = ids[:SOURCE_MAX_LENGTH]
    ids = pad_sequences([ids], SOURCE_MAX_LENGTH)[0]
    val_source_ids.append(ids)

print("Number of validation source sequences:", len(val_source_ids))
print("Length of first sequence:", len(val_source_ids[0]))

Number of validation source sequences: 100
Length of first sequence: 512


### Now Encoding Validation Target:-

In [56]:
# Encoding validation target sequences

val_target_ids = []

for text in val_df["target"]:
    ids = encode_text(text)
    ids = ids[:TARGET_MAX_LENGTH]
    ids = pad_sequences([ids], TARGET_MAX_LENGTH)[0]
    val_target_ids.append(ids)

print("Number of validation target sequences:", len(val_target_ids))
print("Length of first target sequence:", len(val_target_ids[0]))

Number of validation target sequences: 100
Length of first target sequence: 128


- Dataset	                     -   Source	               -   Target
- Train	                         -  800 × 512	           -  800 × 128
- Validation	                 -  100 × 512	           -  100 × 128
- Test	                         -    100                  -   -

### Now For Test Source Sequence:-
- same process

In [57]:
# Encoding test source sequences

test_source_ids = []

for text in test_df["source"]:
    ids = encode_text(text)
    ids = ids[:SOURCE_MAX_LENGTH]
    ids = pad_sequences([ids], SOURCE_MAX_LENGTH)[0]
    test_source_ids.append(ids)

print("Number of test source sequences:", len(test_source_ids))
print("Length of first sequence:", len(test_source_ids[0]))

Number of test source sequences: 100
Length of first sequence: 512


In [58]:
# Encoding test target sequences

test_target_ids = []

for text in test_df["target"]:
    ids = encode_text(text)
    ids = ids[:TARGET_MAX_LENGTH]
    ids = pad_sequences([ids], TARGET_MAX_LENGTH)[0]
    test_target_ids.append(ids)

print("Number of test target sequences:", len(test_target_ids))
print("Length of first target sequence:", len(test_target_ids[0]))

Number of test target sequences: 100
Length of first target sequence: 128


### The Transformer will operate on tensors, not ordinary Python lists. Tensors allow PyTorch to perform operations efficiently and eventually run        them through the neural network.

In [59]:
# Converting training sequences to PyTorch tensors

import torch

train_source_tensor = torch.tensor(train_source_ids, dtype=torch.long)
train_target_tensor = torch.tensor(train_target_ids, dtype=torch.long)

print("Train source tensor shape:", train_source_tensor.shape)
print("Train target tensor shape:", train_target_tensor.shape)

print("\nTrain source tensor dtype:", train_source_tensor.dtype)
print("Train target tensor dtype:", train_target_tensor.dtype)

Train source tensor shape: torch.Size([800, 512])
Train target tensor shape: torch.Size([800, 128])

Train source tensor dtype: torch.int64
Train target tensor dtype: torch.int64


In [60]:
#Pipeline for now:--
#Text to Tokens to Token IDs to Padding/Truncation to PyTorch Tensors

In [61]:
#also converting validation and test data to tensors we need to convert it b.c it is also a python list and model dont understand the python list so we have to covert it 

In [62]:
# Converting validation and test sequences to PyTorch tensors

val_source_tensor = torch.tensor(val_source_ids, dtype=torch.long)
val_target_tensor = torch.tensor(val_target_ids, dtype=torch.long)

test_source_tensor = torch.tensor(test_source_ids, dtype=torch.long)
test_target_tensor = torch.tensor(test_target_ids, dtype=torch.long)

print("Validation source tensor shape:", val_source_tensor.shape)
print("Validation target tensor shape:", val_target_tensor.shape)

print("\nTest source tensor shape:", test_source_tensor.shape)
print("Test target tensor shape:", test_target_tensor.shape)

Validation source tensor shape: torch.Size([100, 512])
Validation target tensor shape: torch.Size([100, 128])

Test source tensor shape: torch.Size([100, 512])
Test target tensor shape: torch.Size([100, 128])


In [63]:
# Now we will create a pytorch dataset:

In [64]:
# Creating a PyTorch Dataset

from torch.utils.data import Dataset

class PubMedQADataset(Dataset):
    
    def __init__(self, source_tensor, target_tensor):
        self.source = source_tensor
        self.target = target_tensor

    def __len__(self):
        return len(self.source)

    def __getitem__(self, index):
        return self.source[index], self.target[index]

In [65]:
#After creating the pytorch dataset now we have 
#train source tensor = [800, 512]
#train target tensor = [800, 128]
#We need a struture which says source 32 belongs with target 32 
#why dataset and data loader its like a restruatent has a kitchen/storage

In [66]:
# Creating the training Dataset

train_dataset = PubMedQADataset(
    train_source_tensor,
    train_target_tensor
)

print("Training dataset created successfully.")
print("Number of training examples:", len(train_dataset))

Training dataset created successfully.
Number of training examples: 800


In [67]:
# Inspecting one example from the training Dataset

sample_source, sample_target = train_dataset[0]

print("Source shape:", sample_source.shape)
print("Target shape:", sample_target.shape)

print("\nFirst 20 source token IDs:")
print(sample_source[:20])

print("\nFirst 20 target token IDs:")
print(sample_target[:20])

Source shape: torch.Size([512])
Target shape: torch.Size([128])

First 20 source token IDs:
tensor([ 5,  6,  1,  7,  8,  9, 10, 11, 12, 13, 14,  6, 15, 16, 17, 18,  7,  8,
        19, 20])

First 20 target token IDs:
tensor([  25,    9,   10,   11,   12,   13,   14,  107,  108,  109,   44,  110,
         111,   23,   44,  112, 7836,   42, 1326,   31])


In [68]:
#How do we give the Transformer multiple examples at once that why we are now creating a dataloader
#dataloader will give batches to the transformer like batch size 16 (16 source input+16 target input)

In [69]:
# Creating the DataLoader for training

from torch.utils.data import DataLoader

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Training DataLoader created successfully.")
print("Batch size:", BATCH_SIZE)
print("Number of batches:", len(train_loader))

Training DataLoader created successfully.
Batch size: 16
Number of batches: 50


In [70]:
#Now we have
#800 training examples
#        ↓
#Dataset
#        ↓
#DataLoader
#        ↓
#50 batches

In [71]:
# Checking from the training DataLoader

source_batch, target_batch = next(iter(train_loader))

print("Source batch shape:", source_batch.shape)
print("Target batch shape:", target_batch.shape)

print("\nSource batch dtype:", source_batch.dtype)
print("Target batch dtype:", target_batch.dtype)

Source batch shape: torch.Size([16, 512])
Target batch shape: torch.Size([16, 128])

Source batch dtype: torch.int64
Target batch dtype: torch.int64


In [72]:
#Text to Tokens to Token IDs to Padding/Truncation to Tensors to Dataset to DataLoader to Batch


## now we will do Same with Validation and test set

In [73]:
# Creating validation and test datasets

val_dataset = PubMedQADataset(
    val_source_tensor,
    val_target_tensor
)

test_dataset = PubMedQADataset(
    test_source_tensor,
    test_target_tensor
)

print("Validation dataset created successfully.")
print("Number of validation examples:", len(val_dataset))

print("\nTest dataset created successfully.")
print("Number of test examples:", len(test_dataset))

Validation dataset created successfully.
Number of validation examples: 100

Test dataset created successfully.
Number of test examples: 100


### We used shuffle=True for training because we want the model to see training examples in a different order
### But for validation and test shuffle=False because there is no learning happening—we simply want consistent evaluation

In [74]:
# Creating validation and test DataLoaders

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Validation DataLoader created successfully.")
print("Number of validation batches:", len(val_loader))

print("\nTest DataLoader created successfully.")
print("Number of test batches:", len(test_loader))

Validation DataLoader created successfully.
Number of validation batches: 7

Test DataLoader created successfully.
Number of test batches: 7


In [75]:
# Checking validation and test batches

val_source_batch, val_target_batch = next(iter(val_loader))
test_source_batch, test_target_batch = next(iter(test_loader))

print("Validation source batch shape:", val_source_batch.shape)
print("Validation target batch shape:", val_target_batch.shape)

print("\nTest source batch shape:", test_source_batch.shape)
print("Test target batch shape:", test_target_batch.shape)

Validation source batch shape: torch.Size([16, 512])
Validation target batch shape: torch.Size([16, 128])

Test source batch shape: torch.Size([16, 512])
Test target batch shape: torch.Size([16, 128])


In [76]:
# Checking special token IDs

PAD_ID = token_to_id["<PAD>"]
UNK_ID = token_to_id["<UNK>"]
BOS_ID = token_to_id["<BOS>"]
EOS_ID = token_to_id["<EOS>"]
SEP_ID = token_to_id["<SEP>"]

print("PAD ID:", PAD_ID)
print("UNK ID:", UNK_ID)
print("BOS ID:", BOS_ID)
print("EOS ID:", EOS_ID)
print("SEP ID:", SEP_ID)

PAD ID: 0
UNK ID: 1
BOS ID: 2
EOS ID: 3
SEP ID: 4


In [77]:
# Why we are doing this beacuse transformer dont know when to start and when to end that why BOS EOS is using begining and end of sequence.
# Example <BOS> The study found that....... <EOS>
# We have to tell it that its start and its end 
# after that we will split it decoder input <BOS> and excepted output <EOS>
# text to tokens to BOS/EOS to decoder input to expected output

In [78]:
# Inspecting one target answer before adding BOS and EOS

sample_target_text = train_df["target"].iloc[0]

sample_target_tokens = tokenize(sample_target_text)

print("Original target text:")
print(sample_target_text)

print("\nFirst 20 target tokens:")
print(sample_target_tokens[:20])

print("\nNumber of target tokens:", len(sample_target_tokens))

Original target text:
VAT area measurement at a single level 3 cm above the lower margin of the L3 vertebra is feasible and can reflect changes in VAT volume and body weight. Advances in knowledge: As VAT area at a CT slice 3cm above the lower margin of L3 can best reflect interval changes in VAT volume and body weight, VAT area measurement should be selected at this location.

First 20 target tokens:
['VAT', 'area', 'measurement', 'at', 'a', 'single', 'level', '3', 'cm', 'above', 'the', 'lower', 'margin', 'of', 'the', 'L3', 'vertebra', 'is', 'feasible', 'and']

Number of target tokens: 70


In [79]:
# Now decoder need clear starting which is <BOS> VAT ..............<EOS>

In [80]:
# Adding BOS and EOS to one target sequence

sample_target_with_specials = (
    [BOS_ID]
    + [
        token_to_id.get(token, UNK_ID)
        for token in sample_target_tokens
    ]
    + [EOS_ID]
)

print("Target sequence with BOS and EOS:")
print(sample_target_with_specials)

print("\nOriginal token count:", len(sample_target_tokens))
print("New sequence length:", len(sample_target_with_specials))

Target sequence with BOS and EOS:
[2, 25, 9, 10, 11, 12, 13, 14, 107, 108, 109, 44, 110, 111, 23, 44, 112, 7836, 42, 1326, 31, 15, 50, 51, 52, 25, 19, 31, 53, 54, 37, 1, 52, 2429, 6, 2420, 25, 9, 11, 12, 45, 46, 1, 109, 44, 110, 111, 23, 112, 15, 49, 50, 85, 51, 52, 25, 19, 31, 53, 54, 71, 25, 9, 10, 2507, 29, 1099, 11, 40, 47, 37, 3]

Original token count: 70
New sequence length: 72


In [81]:
# now we can clearly see that we have created it at start 2 which is BOS and in end 3 which is EOS 

In [82]:
# Demonstrating the decoder input and expected output

decoder_input = sample_target_with_specials[:-1]
decoder_target = sample_target_with_specials[1:]

print("Decoder input:")
print(decoder_input)

print("\nExpected output:")
print(decoder_target)

print("\nDecoder input length:", len(decoder_input))
print("Expected output length:", len(decoder_target))

Decoder input:
[2, 25, 9, 10, 11, 12, 13, 14, 107, 108, 109, 44, 110, 111, 23, 44, 112, 7836, 42, 1326, 31, 15, 50, 51, 52, 25, 19, 31, 53, 54, 37, 1, 52, 2429, 6, 2420, 25, 9, 11, 12, 45, 46, 1, 109, 44, 110, 111, 23, 112, 15, 49, 50, 85, 51, 52, 25, 19, 31, 53, 54, 71, 25, 9, 10, 2507, 29, 1099, 11, 40, 47, 37]

Expected output:
[25, 9, 10, 11, 12, 13, 14, 107, 108, 109, 44, 110, 111, 23, 44, 112, 7836, 42, 1326, 31, 15, 50, 51, 52, 25, 19, 31, 53, 54, 37, 1, 52, 2429, 6, 2420, 25, 9, 11, 12, 45, 46, 1, 109, 44, 110, 111, 23, 112, 15, 49, 50, 85, 51, 52, 25, 19, 31, 53, 54, 71, 25, 9, 10, 2507, 29, 1099, 11, 40, 47, 37, 3]

Decoder input length: 71
Expected output length: 71


In [83]:
# We can clearly see decoder input length 71 and excepted output lenth 71 what happend here :- teacher forcing
#teacher forcing: during training, 
#we give the decoder the correct previous tokens rather than making it use its own possibly-wrong predictions.
#eg:- Teacher gives <BOS> model will understand it like VAT then teacher give <BOS> VAT model predict area then <BOS> VAT area model should predict measured

## Creating it for all 800 training examples

In [84]:
# now we will create a function that 
# convert answer into token
# Convert each token into vocab id
# handle words which are not present in the vocab
# Leave 2 space for BOS EOS
# creates the decoder-ready target struture

In [85]:
def prepare_target(text):
    """
    Convert target text into token IDs with BOS and EOS.
    Reserve 2 positions for BOS and EOS.
    """
    tokens = tokenize(text)

    token_ids = [
        token_to_id.get(token, UNK_ID)
        for token in tokens
    ]

    # Reserve 2 positions for BOS and EOS
    token_ids = token_ids[:TARGET_MAX_LENGTH - 2]

    return [BOS_ID] + token_ids + [EOS_ID]

In [86]:
sample_prepared_target = prepare_target(train_df["target"].iloc[0])

print("Prepared target:")
print(sample_prepared_target)

print("\nFirst token ID:", sample_prepared_target[0])
print("Last token ID:", sample_prepared_target[-1])
print("Sequence length:", len(sample_prepared_target))

print("\nExpected:")
print("First token should be BOS:", BOS_ID)
print("Last token should be EOS:", EOS_ID)
print("Maximum length:", TARGET_MAX_LENGTH)

Prepared target:
[2, 25, 9, 10, 11, 12, 13, 14, 107, 108, 109, 44, 110, 111, 23, 44, 112, 7836, 42, 1326, 31, 15, 50, 51, 52, 25, 19, 31, 53, 54, 37, 1, 52, 2429, 6, 2420, 25, 9, 11, 12, 45, 46, 1, 109, 44, 110, 111, 23, 112, 15, 49, 50, 85, 51, 52, 25, 19, 31, 53, 54, 71, 25, 9, 10, 2507, 29, 1099, 11, 40, 47, 37, 3]

First token ID: 2
Last token ID: 3
Sequence length: 72

Expected:
First token should be BOS: 2
Last token should be EOS: 3
Maximum length: 128


In [87]:
# Now we can see first token 2 and last token 3

In [88]:
#Prepare all training targets:
train_target_with_specials = []

for text in train_df["target"]:
    prepared_target = prepare_target(text)
    train_target_with_specials.append(prepared_target)

print("Number of prepared training targets:", len(train_target_with_specials))

print("\nFirst training target:")
print(train_target_with_specials[0])

print("\nLength of first training target:", len(train_target_with_specials[0]))

Number of prepared training targets: 800

First training target:
[2, 25, 9, 10, 11, 12, 13, 14, 107, 108, 109, 44, 110, 111, 23, 44, 112, 7836, 42, 1326, 31, 15, 50, 51, 52, 25, 19, 31, 53, 54, 37, 1, 52, 2429, 6, 2420, 25, 9, 11, 12, 45, 46, 1, 109, 44, 110, 111, 23, 112, 15, 49, 50, 85, 51, 52, 25, 19, 31, 53, 54, 71, 25, 9, 10, 2507, 29, 1099, 11, 40, 47, 37, 3]

Length of first training target: 72


In [89]:
# we are doing padding to make the shape consistent

In [90]:
train_target_padded = pad_sequences(
    train_target_with_specials,
    TARGET_MAX_LENGTH
)

print("Number of padded training targets:", len(train_target_padded))
print("Length of first target:", len(train_target_padded[0]))

print("\nFirst 20 IDs:")
print(train_target_padded[0][:20])

print("\nLast 10 IDs:")
print(train_target_padded[0][-10:])

Number of padded training targets: 800
Length of first target: 128

First 20 IDs:
[2, 25, 9, 10, 11, 12, 13, 14, 107, 108, 109, 44, 110, 111, 23, 44, 112, 7836, 42, 1326]

Last 10 IDs:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [91]:
# Decoder Input + Decoder Target transformer will predict the next word

In [92]:
#Creating decoder input and expected output for one now:

In [93]:
sample_target = train_target_padded[0]

sample_decoder_input = sample_target[:-1]
sample_decoder_target = sample_target[1:]

print("Decoder input:")
print(sample_decoder_input)

print("\nExpected output:")
print(sample_decoder_target)

print("\nDecoder input length:", len(sample_decoder_input))
print("Expected output length:", len(sample_decoder_target))

Decoder input:
[2, 25, 9, 10, 11, 12, 13, 14, 107, 108, 109, 44, 110, 111, 23, 44, 112, 7836, 42, 1326, 31, 15, 50, 51, 52, 25, 19, 31, 53, 54, 37, 1, 52, 2429, 6, 2420, 25, 9, 11, 12, 45, 46, 1, 109, 44, 110, 111, 23, 112, 15, 49, 50, 85, 51, 52, 25, 19, 31, 53, 54, 71, 25, 9, 10, 2507, 29, 1099, 11, 40, 47, 37, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Expected output:
[25, 9, 10, 11, 12, 13, 14, 107, 108, 109, 44, 110, 111, 23, 44, 112, 7836, 42, 1326, 31, 15, 50, 51, 52, 25, 19, 31, 53, 54, 37, 1, 52, 2429, 6, 2420, 25, 9, 11, 12, 45, 46, 1, 109, 44, 110, 111, 23, 112, 15, 49, 50, 85, 51, 52, 25, 19, 31, 53, 54, 71, 25, 9, 10, 2507, 29, 1099, 11, 40, 47, 37, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Decoder input length: 127
Expe

In [94]:
#Create decoder input and target for all 800

In [95]:
train_decoder_input_ids = []
train_decoder_target_ids = []

for target in train_target_padded:
    decoder_input = target[:-1]
    decoder_target = target[1:]

    train_decoder_input_ids.append(decoder_input)
    train_decoder_target_ids.append(decoder_target)

print("Number of decoder input sequences:", len(train_decoder_input_ids))
print("Number of decoder target sequences:", len(train_decoder_target_ids))

print("\nLength of first decoder input:", len(train_decoder_input_ids[0]))
print("Length of first decoder target:", len(train_decoder_target_ids[0]))

Number of decoder input sequences: 800
Number of decoder target sequences: 800

Length of first decoder input: 127
Length of first decoder target: 127


In [96]:
# now Converting decoder data to tensors

In [97]:
train_decoder_input_tensor = torch.tensor(
    train_decoder_input_ids,
    dtype=torch.long
)

train_decoder_target_tensor = torch.tensor(
    train_decoder_target_ids,
    dtype=torch.long
)

print("Decoder input tensor shape:", train_decoder_input_tensor.shape)
print("Decoder target tensor shape:", train_decoder_target_tensor.shape)

print("\nDecoder input dtype:", train_decoder_input_tensor.dtype)
print("Decoder target dtype:", train_decoder_target_tensor.dtype)

Decoder input tensor shape: torch.Size([800, 127])
Decoder target tensor shape: torch.Size([800, 127])

Decoder input dtype: torch.int64
Decoder target dtype: torch.int64


## source, decoder_input, decoder_target So now we will create a new Dataset specifically for Transformer training

In [98]:
#Create the Transformer Dataset:
class TransformerDataset(Dataset):

    def __init__(self, source_tensor, decoder_input_tensor, decoder_target_tensor):
        self.source = source_tensor
        self.decoder_input = decoder_input_tensor
        self.decoder_target = decoder_target_tensor

    def __len__(self):
        return len(self.source)

    def __getitem__(self, index):
        return (
            self.source[index],
            self.decoder_input[index],
            self.decoder_target[index]
        )


print("Transformer Dataset class created successfully.")

Transformer Dataset class created successfully.


In [99]:
#Create the training Transformer dataset:
train_transformer_dataset = TransformerDataset(
    train_source_tensor,
    train_decoder_input_tensor,
    train_decoder_target_tensor
)

print("Training Transformer Dataset created successfully.")
print("Number of training examples:", len(train_transformer_dataset))

Training Transformer Dataset created successfully.
Number of training examples: 800


In [100]:
#Checking one Transformer example:
sample_source, sample_decoder_input, sample_decoder_target = train_transformer_dataset[0]

print("Source shape:", sample_source.shape)
print("Decoder input shape:", sample_decoder_input.shape)
print("Decoder target shape:", sample_decoder_target.shape)

print("\nFirst 10 source IDs:")
print(sample_source[:10])

print("\nFirst 10 decoder input IDs:")
print(sample_decoder_input[:10])

print("\nFirst 10 decoder target IDs:")
print(sample_decoder_target[:10])

print("\nDecoder input first ID:", sample_decoder_input[0].item())
print("Expected BOS ID:", BOS_ID)

Source shape: torch.Size([512])
Decoder input shape: torch.Size([127])
Decoder target shape: torch.Size([127])

First 10 source IDs:
tensor([ 5,  6,  1,  7,  8,  9, 10, 11, 12, 13])

First 10 decoder input IDs:
tensor([  2,  25,   9,  10,  11,  12,  13,  14, 107, 108])

First 10 decoder target IDs:
tensor([ 25,   9,  10,  11,  12,  13,  14, 107, 108, 109])

Decoder input first ID: 2
Expected BOS ID: 2


In [101]:
#Validation and Test targets:


In [102]:
#Preparing validation targets:-
val_target_with_specials = []

for text in val_df["target"]:
    prepared_target = prepare_target(text)
    val_target_with_specials.append(prepared_target)

val_target_padded = pad_sequences(
    val_target_with_specials,
    TARGET_MAX_LENGTH
)

print("Number of validation targets:", len(val_target_padded))
print("Length of first validation target:", len(val_target_padded[0]))

Number of validation targets: 100
Length of first validation target: 128


In [103]:
#Create validation decoder input and target
val_decoder_input_ids = []
val_decoder_target_ids = []

for target in val_target_padded:
    decoder_input = target[:-1]
    decoder_target = target[1:]

    val_decoder_input_ids.append(decoder_input)
    val_decoder_target_ids.append(decoder_target)

print("Number of validation decoder inputs:", len(val_decoder_input_ids))
print("Number of validation decoder targets:", len(val_decoder_target_ids))

print("\nLength of first decoder input:", len(val_decoder_input_ids[0]))
print("Length of first decoder target:", len(val_decoder_target_ids[0]))

Number of validation decoder inputs: 100
Number of validation decoder targets: 100

Length of first decoder input: 127
Length of first decoder target: 127


In [104]:
#Prepare test targets
test_target_with_specials = []

for text in test_df["target"]:
    prepared_target = prepare_target(text)
    test_target_with_specials.append(prepared_target)

test_target_padded = pad_sequences(
    test_target_with_specials,
    TARGET_MAX_LENGTH
)

print("Number of test targets:", len(test_target_padded))
print("Length of first test target:", len(test_target_padded[0]))

Number of test targets: 100
Length of first test target: 128


In [105]:
#Create test decoder input and target
test_decoder_input_ids = []
test_decoder_target_ids = []

for target in test_target_padded:
    decoder_input = target[:-1]
    decoder_target = target[1:]

    test_decoder_input_ids.append(decoder_input)
    test_decoder_target_ids.append(decoder_target)

print("Number of test decoder inputs:", len(test_decoder_input_ids))
print("Number of test decoder targets:", len(test_decoder_target_ids))

print("\nLength of first decoder input:", len(test_decoder_input_ids[0]))
print("Length of first decoder target:", len(test_decoder_target_ids[0]))

Number of test decoder inputs: 100
Number of test decoder targets: 100

Length of first decoder input: 127
Length of first decoder target: 127


In [106]:
#Convert validation/test decoder data to tensors
val_decoder_input_tensor = torch.tensor(
    val_decoder_input_ids,
    dtype=torch.long
)

val_decoder_target_tensor = torch.tensor(
    val_decoder_target_ids,
    dtype=torch.long
)

test_decoder_input_tensor = torch.tensor(
    test_decoder_input_ids,
    dtype=torch.long
)

test_decoder_target_tensor = torch.tensor(
    test_decoder_target_ids,
    dtype=torch.long
)

print("Validation decoder input shape:", val_decoder_input_tensor.shape)
print("Validation decoder target shape:", val_decoder_target_tensor.shape)

print("\nTest decoder input shape:", test_decoder_input_tensor.shape)
print("Test decoder target shape:", test_decoder_target_tensor.shape)

Validation decoder input shape: torch.Size([100, 127])
Validation decoder target shape: torch.Size([100, 127])

Test decoder input shape: torch.Size([100, 127])
Test decoder target shape: torch.Size([100, 127])


In [107]:
#TRAIN
#800 × 512   Source
#800 × 127   Decoder Input
#800 × 127   Decoder Target

#VALIDATION
#100 × 512   Source
#100 × 127   Decoder Input
#100 × 127   Decoder Target

#TEST
#100 × 512   Source
#100 × 127   Decoder Input
#100 × 127   Decoder Target

In [108]:
#Create validation and test datasets
val_transformer_dataset = TransformerDataset(
    val_source_tensor,
    val_decoder_input_tensor,
    val_decoder_target_tensor
)

test_transformer_dataset = TransformerDataset(
    test_source_tensor,
    test_decoder_input_tensor,
    test_decoder_target_tensor
)

print("Validation Transformer Dataset created successfully.")
print("Number of validation examples:", len(val_transformer_dataset))

print("\nTest Transformer Dataset created successfully.")
print("Number of test examples:", len(test_transformer_dataset))

Validation Transformer Dataset created successfully.
Number of validation examples: 100

Test Transformer Dataset created successfully.
Number of test examples: 100


In [109]:
#Final Transformer DataLoaders
train_transformer_loader = DataLoader(
    train_transformer_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_transformer_loader = DataLoader(
    val_transformer_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_transformer_loader = DataLoader(
    test_transformer_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training Transformer DataLoader:", len(train_transformer_loader), "batches")
print("Validation Transformer DataLoader:", len(val_transformer_loader), "batches")
print("Test Transformer DataLoader:", len(test_transformer_loader), "batches")

Training Transformer DataLoader: 50 batches
Validation Transformer DataLoader: 7 batches
Test Transformer DataLoader: 7 batches


In [110]:
#checking the padding positions
source_batch, decoder_input_batch, decoder_target_batch = next(
    iter(train_transformer_loader)
)

print("Source batch shape:", source_batch.shape)
print("Decoder input batch shape:", decoder_input_batch.shape)
print("Decoder target batch shape:", decoder_target_batch.shape)

print("\nPAD ID:", PAD_ID)

print("\nNumber of PAD tokens in first source sequence:")
print((source_batch[0] == PAD_ID).sum().item())


Source batch shape: torch.Size([16, 512])
Decoder input batch shape: torch.Size([16, 127])
Decoder target batch shape: torch.Size([16, 127])

PAD ID: 0

Number of PAD tokens in first source sequence:
402


In [111]:
#Create the source padding mask
source_padding_mask = source_batch == PAD_ID

print("Source padding mask shape:", source_padding_mask.shape)

print("\nFirst 30 mask values:")
print(source_padding_mask[0, :30])

print("\nLast 30 mask values:")
print(source_padding_mask[0, -30:])

print("\nNumber of masked positions in first sequence:")
print(source_padding_mask[0].sum().item())

Source padding mask shape: torch.Size([16, 512])

First 30 mask values:
tensor([False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False])

Last 30 mask values:
tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True])

Number of masked positions in first sequence:
402


In [112]:
#Target padding mask
target_padding_mask = decoder_input_batch == PAD_ID

print("Target padding mask shape:", target_padding_mask.shape)

print("\nFirst 30 mask values:")
print(target_padding_mask[0, :30])

print("\nLast 30 mask values:")
print(target_padding_mask[0, -30:])

print("\nNumber of masked positions in first sequence:")
print(target_padding_mask[0].sum().item())

Target padding mask shape: torch.Size([16, 127])

First 30 mask values:
tensor([False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False])

Last 30 mask values:
tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True])

Number of masked positions in first sequence:
46


In [113]:
#Create the causal mask
target_length = decoder_input_batch.size(1)

causal_mask = torch.triu(
    torch.ones(target_length, target_length, dtype=torch.bool),
    diagonal=1
)

print("Causal mask shape:", causal_mask.shape)

print("\nFirst 10 × 10 positions:")
print(causal_mask[:10, :10])

Causal mask shape: torch.Size([127, 127])

First 10 × 10 positions:
tensor([[False,  True,  True,  True,  True,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True,  True,  True,  True,  True],
        [False, False, False, False,  True,  True,  True,  True,  True,  True],
        [False, False, False, False, False,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False,  True,  True,  True,  True],
        [False, False, False, False, False, False, False,  True,  True,  True],
        [False, False, False, False, False, False, False, False,  True,  True],
        [False, False, False, False, False, False, False, False, False,  True],
        [False, False, False, False, False, False, False, False, False, False]])


In [121]:
#saving the files

In [122]:
import json
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

with open(processed_dir / "vocabulary.json", "w", encoding="utf-8") as f:
    json.dump(token_to_id, f, ensure_ascii=False, indent=2)

print("Vocabulary saved successfully.")
print("Path:", processed_dir / "vocabulary.json")

Vocabulary saved successfully.
Path: ..\data\processed\vocabulary.json


In [123]:
train_data = {
    "source": train_source_tensor,
    "decoder_input": train_decoder_input_tensor,
    "decoder_target": train_decoder_target_tensor
}

torch.save(
    train_data,
    processed_dir / "train_tensors.pt"
)

print("Training tensors saved successfully.")
print("Path:", processed_dir / "train_tensors.pt")

Training tensors saved successfully.
Path: ..\data\processed\train_tensors.pt


In [124]:
val_data = {
    "source": val_source_tensor,
    "decoder_input": val_decoder_input_tensor,
    "decoder_target": val_decoder_target_tensor
}

torch.save(
    val_data,
    processed_dir / "val_tensors.pt"
)

print("Validation tensors saved successfully.")
print("Path:", processed_dir / "val_tensors.pt")

Validation tensors saved successfully.
Path: ..\data\processed\val_tensors.pt


In [125]:
test_data = {
    "source": test_source_tensor,
    "decoder_input": test_decoder_input_tensor,
    "decoder_target": test_decoder_target_tensor
}

torch.save(
    test_data,
    processed_dir / "test_tensors.pt"
)

print("Test tensors saved successfully.")
print("Path:", processed_dir / "test_tensors.pt")

Test tensors saved successfully.
Path: ..\data\processed\test_tensors.pt


In [126]:
#verify
print("Checking saved files:")

print("Vocabulary:", (processed_dir / "vocabulary.json").exists())
print("Train tensors:", (processed_dir / "train_tensors.pt").exists())
print("Validation tensors:", (processed_dir / "val_tensors.pt").exists())
print("Test tensors:", (processed_dir / "test_tensors.pt").exists())

Checking saved files:
Vocabulary: True
Train tensors: True
Validation tensors: True
Test tensors: True


In [114]:
#Transformer hyperparameters
VOCAB_SIZE = len(token_to_id)

D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2

D_FF = 512
DROPOUT = 0.1

print("Vocabulary size:", VOCAB_SIZE)
print("Model dimension:", D_MODEL)
print("Number of attention heads:", NUM_HEADS)
print("Encoder layers:", NUM_ENCODER_LAYERS)
print("Decoder layers:", NUM_DECODER_LAYERS)
print("Feed-forward dimension:", D_FF)
print("Dropout:", DROPOUT)

Vocabulary size: 9537
Model dimension: 128
Number of attention heads: 4
Encoder layers: 2
Decoder layers: 2
Feed-forward dimension: 512
Dropout: 0.1


In [115]:
# What it means:--

#Vocabulary = 9,537 tokens
#        ↓
#Embedding
#        ↓
#128-dimensional representation
#        ↓
#4 attention heads
#        ↓
#32 dimensions per head

## EMbedding

### embedding layer converts: token ID to learned vector

In [116]:
#Creating the embedding layer
import torch.nn as nn

embedding = nn.Embedding(
    num_embeddings=VOCAB_SIZE,
    embedding_dim=D_MODEL,
    padding_idx=PAD_ID
)

print("Embedding layer created successfully.")
print("Embedding matrix shape:", embedding.weight.shape)

Embedding layer created successfully.
Embedding matrix shape: torch.Size([9537, 128])


In [117]:
#Test source embeddings
source_embeddings = embedding(source_batch)

print("Source token IDs shape:", source_batch.shape)
print("Source embedding shape:", source_embeddings.shape)

print("\nFirst token ID:", source_batch[0, 0].item())
print("First token embedding shape:", source_embeddings[0, 0].shape)

Source token IDs shape: torch.Size([16, 512])
Source embedding shape: torch.Size([16, 512, 128])

First token ID: 5
First token embedding shape: torch.Size([128])


### Positional Encoding eg:- My name is anuj ---- anuj my name is 


In [118]:
#Create positional encoding
import math

def create_positional_encoding(max_length, d_model):
    """
    Create sinusoidal positional encodings.

    Shape:
        [max_length, d_model]
    """

    position = torch.arange(max_length).unsqueeze(1)

    div_term = torch.exp(
        torch.arange(0, d_model, 2) *
        (-math.log(10000.0) / d_model)
    )

    pe = torch.zeros(max_length, d_model)

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    return pe


print("Positional encoding function created successfully.")

Positional encoding function created successfully.


In [119]:
#Generate positional encodings
positional_encoding = create_positional_encoding(
    max_length=SOURCE_MAX_LENGTH,
    d_model=D_MODEL
)

print("Positional encoding shape:", positional_encoding.shape)

print("\nFirst position:")
print(positional_encoding[0])

print("\nSecond position:")
print(positional_encoding[1])

Positional encoding shape: torch.Size([512, 128])

First position:
tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1., 0., 1.,
        0., 1.])

Second position:
tensor([8.4147e-01, 5.4030e-01, 7.6172e-01, 6.4791e-01, 6.8156e-01, 7.3176e-01,
        6.0469e-01, 7.9646e-01, 5.3317e-01, 8.4601e-01, 4.6795e-01, 8.8376e-01,
        4.0931e-01, 9.1240e-01, 3.5711e-01, 9.3406e-01, 3.1098e-01, 9.5042e-01,
        2.7043e-01, 9.6274e-01, 2.3492e-01, 9.7201e-01, 2.0391e-01, 9.7899e-01,
        1.7689e-01

In [120]:
#Combine embeddings + positional encoding
source_embeddings_with_position = (
    source_embeddings + positional_encoding
)

print("Source embeddings shape:", source_embeddings.shape)
print("Positional encoding shape:", positional_encoding.shape)
print("Combined representation shape:", source_embeddings_with_position.shape)

Source embeddings shape: torch.Size([16, 512, 128])
Positional encoding shape: torch.Size([512, 128])
Combined representation shape: torch.Size([16, 512, 128])
